In [3]:
import re
import random
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from sklearn.model_selection import train_test_split

## Load dataset (OpenSubtitles via HuggingFace)

In [4]:
dataset = load_dataset("opus100", "en-ru", split="train[:100000]")
dataset

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


test-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/124M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/310k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['translation'],
    num_rows: 100000
})

## Extract sentences

In [5]:
data = []

for item in dataset:
    en = item["translation"]["en"]
    ru = item["translation"]["ru"]
    data.append((en, ru))

print("Total pairs:", len(data))

Total pairs: 100000


## Create synthetic code-switched sentences

In [6]:
def mix_sentences(en, ru):
    en_words = en.split()
    ru_words = ru.split()
    
    if len(en_words) < 3 or len(ru_words) < 3:
        return None
    
    # random cut points
    cut_en = random.randint(1, len(en_words)-1)
    cut_ru = random.randint(1, len(ru_words)-1)
    
    # mix
    mixed = en_words[:cut_en] + ru_words[cut_ru:]
    
    return " ".join(mixed)

## Build dataset

In [7]:
mixed_sentences = []

for en, ru in tqdm(data):
    mixed = mix_sentences(en, ru)
    if mixed:
        mixed_sentences.append(mixed)

print("Mixed samples:", len(mixed_sentences))

100%|██████████| 100000/100000 [00:00<00:00, 364665.96it/s]

Mixed samples: 79996


## Language detection (token-level)

In [8]:
def detect_lang(token):
    if re.search("[а-яА-Я]", token):
        return "RU"
    else:
        return "EN"

## Create switch-point labels

In [9]:
def create_labels(sentence):
    tokens = sentence.split()
    langs = [detect_lang(t) for t in tokens]
    
    switch_labels = []
    
    for i in range(len(langs)-1):
        if langs[i] != langs[i+1]:
            switch_labels.append(1)
        else:
            switch_labels.append(0)
    
    return tokens, langs, switch_labels

## Apply to dataset

In [10]:
processed_data = []

for sent in tqdm(mixed_sentences):
    tokens, langs, switches = create_labels(sent)
    
    if len(tokens) < 5:
        continue
        
    processed_data.append({
        "sentence": sent,
        "tokens": tokens,
        "langs": langs,
        "switches": switches
    })

df = pd.DataFrame(processed_data)
print(df.head())

100%|██████████| 79996/79996 [00:00<00:00, 104085.38it/s]

                                            sentence  \
0  The schedule below is tentative; расписание яв...   
1        He'd been out there a few несколько недель.   
2                       It'll make you станет лучше.   
3  My cheekbones and beckoning pelvis already hav...   
4           I see вижу всех этих уродов лгущих тебе.   

                                              tokens  \
0  [The, schedule, below, is, tentative;, расписа...   
1  [He'd, been, out, there, a, few, несколько, не...   
2                 [It'll, make, you, станет, лучше.]   
3  [My, cheekbones, and, beckoning, pelvis, alrea...   
4  [I, see, вижу, всех, этих, уродов, лгущих, тебе.]   

                                               langs  \
0  [EN, EN, EN, EN, EN, RU, RU, RU, RU, RU, RU, R...   
1                   [EN, EN, EN, EN, EN, EN, RU, RU]   
2                               [EN, EN, EN, RU, RU]   
3  [EN, EN, EN, EN, EN, EN, EN, EN, EN, EN, EN, E...   
4                   [EN, EN, RU, RU, RU, RU, R

## Basic EDA

In [11]:
print("Total samples:", len(df))

lengths = df["tokens"].apply(len)
print("Avg length:", lengths.mean())

switch_counts = df["switches"].apply(sum)
print("Avg switches per sentence:", switch_counts.mean())

Total samples: 59271
Avg length: 15.70477974051391
Avg switches per sentence: 1.3111639756373268


## Train / Validation / Test split

In [12]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(len(train_df), len(val_df), len(test_df))

47416 5927 5928


## Save datasets

In [15]:
train_df.to_json("../data/train.json", orient="records", lines=True)
val_df.to_json("../data/val.json", orient="records", lines=True)
test_df.to_json("../data/test.json", orient="records", lines=True)